# Batch processing I22 SAXS/WAXS data using the MoDaCor server

This notebook processes a batch of I22 SAXS/WAXS measurements with the MoDaCor runtime server. It extends the simpler `run_pipeline_job` notebook: use that notebook for a first single-file run, and use this server workflow when many similar files should reuse cached runtime state.

The WAXS pipeline and the SAXS pipeline are instantiated in separate runtime sessions on the server, and configured 
with their respective pipelines. 

This example is currently developed against MoDaCor 1.7.0. The repository README records the development baseline; the first archival release will pin an immutable MoDaCor revision after the complete SAXS/WAXS workflow is rerun. The included DAWN cross-check pipelines deliberately reproduce selected DAWN behaviour, while the `solids_operando` pipelines represent the recommended physical corrections.

Run the cells from top to bottom. For normal use, only edit **User Configuration**.

Fresh environment setup from a terminal:

```bash
git clone https://github.com/BAMresearch/MoDaCor.git
cd MoDaCor
uv venv --python 3.12 .venv
source .venv/bin/activate
uv pip install -e ".[server,attenuation,plotting]" requests matplotlib ipykernel
python -m ipykernel install --user --name modacor-i22 --display-name "Python (MoDaCor I22)"
```

After installing, select the `Python (MoDaCor I22)` kernel for this notebook.


## Optional Notebook Install Cell

Run the next cell only if this notebook kernel cannot import MoDaCor, FastAPI, uvicorn, requests, or matplotlib.


In [ ]:
# Use the terminal setup above when imports fail, then restart the notebook kernel.
# Keeping environment creation outside the notebook makes the selected MoDaCor
# revision explicit and reproducible.


## User Configuration

Start Jupyter from the examples repository root or this instrument directory. The packaged data and pipelines are discovered automatically. The sample discovery is tolerant: if files are still copying, the notebook will warn and still let you preview the pipeline graph and start the server.


In [ ]:
from pathlib import Path

def locate_example_dir(relative_path):
    start = Path.cwd().resolve()
    for parent in (start, *start.parents):
        for candidate in (parent, parent / relative_path):
            if (candidate / "data-manifest.json").is_file() and (candidate / "pipelines").is_dir():
                return candidate
    raise FileNotFoundError(
        "Could not locate DLS/I22. Start Jupyter from the examples repository root "
        "or from the DLS/I22 instrument directory."
    )


PROJECT_DIR = locate_example_dir(Path("DLS") / "I22")
PIPELINE_PATHS = {
    "SAXS": PROJECT_DIR / "pipelines" / "I22_SAXS_solids_operando.yaml",
    "WAXS": PROJECT_DIR / "pipelines" / "I22_WAXS_solids_operando.yaml",
    # "SAXS": PROJECT_DIR / "pipelines" / "I22_SAXS_DAWN_crosscheck.yaml",
    # "WAXS": PROJECT_DIR / "pipelines" / "I22_WAXS_DAWN_crosscheck.yaml",
}

DATA_ROOT = PROJECT_DIR / "data"
EXAMPLE_PROCESSING_DIR = DATA_ROOT / "processing"
CALIBRATION_FILES = {
    "SAXS": EXAMPLE_PROCESSING_DIR / "SAXS_calibration.nxs",
    "WAXS": EXAMPLE_PROCESSING_DIR / "WAXS_calibration.nxs",
}
MASK_FILES = {
    "SAXS": EXAMPLE_PROCESSING_DIR / "SAXS_mask.nxs",
    "WAXS": EXAMPLE_PROCESSING_DIR / "WAXS_mask.nxs",
}

# Select which detector sessions to create and run.
DETECTORS_TO_RUN = ["SAXS", "WAXS"]
DETECTOR = DETECTORS_TO_RUN[0]  # selected detector for single-detector previews
BACKGROUND_FILES = {
    "SAXS": DATA_ROOT / "i22-977723.nxs",
    "WAXS": DATA_ROOT / "i22-977723.nxs",
}

WORK_DIR = PROJECT_DIR / "work"
PREPROCESSED_DATA_DIR = WORK_DIR / "preprocessed"
PREPROCESSED_CALIBRATION_DIR = WORK_DIR / "preprocessed_calibration"
OUTPUT_DIR = WORK_DIR / "output"

PIPELINE_PATH = PIPELINE_PATHS[DETECTOR]
BACKGROUND_FILE = BACKGROUND_FILES[DETECTOR]
SESSION_IDS = {detector: f"i22-{detector.lower()}-server-batch" for detector in DETECTORS_TO_RUN}
SESSION_ID = SESSION_IDS[DETECTOR]
SERVER_HOST = "127.0.0.1"
SEPARATE_SERVER_PER_DETECTOR = False
SERVER_PORTS = {"SAXS": 8901, "WAXS": 8902}
SERVER_PORT = SERVER_PORTS[DETECTOR]
SERVER_LOG_PATHS = {
    detector: OUTPUT_DIR / f"modacor_server_{detector.lower()}.log"
    for detector in DETECTORS_TO_RUN
}
SERVER_LOG_PATH = SERVER_LOG_PATHS[DETECTOR]

SAMPLE_GLOB = "i22-978???.nxs"
BSDIODES_CHANNEL = 1
ABSOLUTE_INTENSITY_FACTOR = 3.8e-15
OVERWRITE_PREPROCESSED = False
MAX_SAMPLES_TO_PROCESS = None

# Set this to False while debugging so the batch stops at the first failed file.
CONTINUE_ON_SAMPLE_ERROR = False
# Use this with SEPARATE_SERVER_PER_DETECTOR to test real process-level concurrency.
PARALLEL_DETECTOR_RUNS = SEPARATE_SERVER_PER_DETECTOR
ROLLBACK_SNAPSHOT = False
RESET_SESSION_AFTER_FAILURE = True

TRACE_ENABLED = True
TRACE_WATCH = {"sample": ["signal"], "background": ["signal"]}

PLOT_SINK_REF = "plots"
RESULT_HDF_SINK_REF = "result_hdf"
LIVE_PLOT_IDS = {
    "SAXS": {"1d": "saxs-1d", "2d": "saxs-2d"},
    "WAXS": {"1d": "waxs-1d", "2d": "waxs-2d"},
}

SAMPLE_OUTPUT_DATA_PATHS = [
    "/sample/signal",
    "/sample/Q",
    # "/sample/Psi",
    # "/sample/Omega",
    # "/sample/pixel_index",
    # "/sample/mask",
]


## Import Checks And File Discovery

This cell checks imports, creates the output directory, and finds sample files. If no sample files are found yet, rerun this cell after copying finishes.


In [ ]:
import atexit
import json
import os
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime

import h5py
import numpy as np
import requests
from IPython.display import JSON, Markdown, display

import modacor

try:
    import hdf5plugin  # noqa: F401
except ImportError:
    print("hdf5plugin is not installed; I22 detector data compressed with Blosc will not load.")
    print("Rerun the install cell after updating MoDaCor, then restart the kernel/server.")

unknown_detectors = set(DETECTORS_TO_RUN) - {"SAXS", "WAXS"}
if unknown_detectors:
    raise ValueError(f"Unknown detector(s): {sorted(unknown_detectors)}")
if DETECTOR not in DETECTORS_TO_RUN:
    raise ValueError("DETECTOR must be included in DETECTORS_TO_RUN.")

print(f"Python: {sys.executable}")
print(f"MoDaCor: {modacor.__version__}")
print(f"MoDaCor module: {modacor.__file__}")

paths_to_check = {
    "PROJECT_DIR": PROJECT_DIR,
    "PIPELINE_PATH_SAXS": PIPELINE_PATHS["SAXS"],
    "PIPELINE_PATH_WAXS": PIPELINE_PATHS["WAXS"],
    "DATA_ROOT": DATA_ROOT,
    "SAXS_CALIBRATION": CALIBRATION_FILES["SAXS"],
    "SAXS_MASK": MASK_FILES["SAXS"],
    "WAXS_CALIBRATION": CALIBRATION_FILES["WAXS"],
    "WAXS_MASK": MASK_FILES["WAXS"],
    "BACKGROUND_FILE": BACKGROUND_FILE,
}
for path_name, path in paths_to_check.items():
    print(f"{path_name}: {path} ({'ok' if path.exists() else 'missing'})")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PREPROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
PREPROCESSED_CALIBRATION_DIR.mkdir(parents=True, exist_ok=True)


def is_sample_candidate(path: Path) -> bool:
    """Return True only for I22 masters with the linked inputs used here."""
    if not path.is_file() or path.resolve() == BACKGROUND_FILE.resolve():
        return False
    stem = path.stem
    required = [
        DATA_ROOT / f"{stem}-Pilatus2M_SAXS.h5",
        DATA_ROOT / f"{stem}-Pilatus2M_WAXS.h5",
        DATA_ROOT / f"{stem}-bsdiodes.h5",
    ]
    return all(candidate.is_file() for candidate in required)


sample_files = sorted(path for path in DATA_ROOT.glob(SAMPLE_GLOB) if is_sample_candidate(path))

print(f"Found {len(sample_files)} sample file(s).")
for index, sample_file in enumerate(sample_files[:10], start=1):
    print(f"{index:02d}: {sample_file}")
if len(sample_files) > 10:
    print(f"... and {len(sample_files) - 10} more")
if not sample_files:
    print("No sample files found yet. Calibration preprocessing and graph preview can still run.")


## Preprocess I22 Inputs For MoDaCor

This cell leaves the raw NeXus/HDF5 files untouched. For each measurement it writes a compact file that externally links `/entry1` from the original master, adds broadcast-ready `(images, frames, 1, 1)` normalization arrays under `/modacor/normalization`, and stores scalar calibration values under `/modacor/calibration`.

The beamstop-diode channel is reduced over its 2,000-sample axis to one mean, standard deviation, SEM, and valid-sample count per `(image, frame)`. Detector count times and transmission are reshaped or broadcast to the same detector-divisor layout.

Detector geometry is not precomputed here. The YAML pipelines point directly at the NeXus calibration files; MoDaCor resolves the detector transformation chains in `PixelCoordinates3D` and `XSGeometryFromPixelCoordinates`.


In [ ]:
PREPROCESSING_VERSION = "2026-09-02-i22-normalization-v3"


def _decode(value):
    if isinstance(value, bytes):
        return value.decode("utf-8")
    if isinstance(value, np.ndarray) and value.shape == ():
        return _decode(value.item())
    return value


def _as_detector_divisor(array):
    array = np.asarray(array)
    return array.reshape(array.shape + (1, 1))


def _measurement_output_path(master_file):
    return PREPROCESSED_DATA_DIR / f"{Path(master_file).stem}_modacor.nxs"


def _needs_rewrite(output_file, *, overwrite):
    output_file = Path(output_file)
    if overwrite or not output_file.exists():
        return True
    try:
        with h5py.File(output_file, "r") as h5:
            return _decode(h5.attrs.get("preprocessing_version", "")) != PREPROCESSING_VERSION
    except OSError:
        return True


def _frame_array(values, leading_shape, *, name):
    values = np.asarray(values, dtype=float)
    if values.shape == leading_shape:
        return values
    if values.size == 1:
        return np.full(leading_shape, float(values.reshape(-1)[0]), dtype=float)
    raise ValueError(f"{name} shape {values.shape} cannot be broadcast to frame shape {leading_shape}.")


def _write_dataset(group, name, values, *, units=None, **attrs):
    dataset = group.create_dataset(name, data=values)
    if units is not None:
        dataset.attrs["units"] = units
    for key, value in attrs.items():
        dataset.attrs[key] = value
    return dataset


def _detector_shapes(source):
    return {
        "SAXS": source["/entry1/detector/data"].shape,
        "WAXS": source["/entry1/Pilatus2M_WAXS/data"].shape,
    }


def preprocess_i22_measurement(master_file, *, overwrite=False):
    """Create one compact MoDaCor-facing I22 measurement file without copying detector images."""
    master_file = Path(master_file).resolve()
    output_file = _measurement_output_path(master_file)
    if not _needs_rewrite(output_file, overwrite=overwrite):
        return output_file

    with h5py.File(master_file, "r") as source:
        bsdiodes = np.asarray(source["/entry1/bsdiodes/data"][()], dtype=float)
        if bsdiodes.ndim != 4 or BSDIODES_CHANNEL >= bsdiodes.shape[-1]:
            raise ValueError(f"Unexpected bsdiodes shape {bsdiodes.shape}; channel {BSDIODES_CHANNEL} is unavailable.")

        detector_shapes = _detector_shapes(source)
        leading_shape = tuple(detector_shapes["SAXS"][:-2])
        if tuple(detector_shapes["WAXS"][:-2]) != leading_shape:
            raise ValueError(f"SAXS/WAXS leading shapes differ: {detector_shapes}")
        if tuple(bsdiodes.shape[:2]) != leading_shape:
            raise ValueError(f"bsdiodes leading shape {bsdiodes.shape[:2]} does not match detector shape {leading_shape}.")

        channel_samples = bsdiodes[..., BSDIODES_CHANNEL]
        valid_count = np.sum(np.isfinite(channel_samples), axis=-1).astype(np.int32)
        mean = np.nanmean(channel_samples, axis=-1)
        std = np.nanstd(channel_samples, axis=-1, ddof=1)
        sem = std / np.sqrt(valid_count)

        count_time_sources = {
            "saxs_count_time": "/entry1/instrument/detector/count_time",
            "waxs_count_time": "/entry1/instrument/Pilatus2M_WAXS/count_time",
        }
        count_times = {}
        for name, path in count_time_sources.items():
            dataset = source[path]
            count_times[name] = (
                _frame_array(dataset[()], leading_shape, name=path),
                str(_decode(dataset.attrs.get("units", "s"))),
            )

        transmission_path = "/entry1/I0/transmission"
        transmission = _frame_array(source[transmission_path][()], leading_shape, name=transmission_path)

    output_file.parent.mkdir(parents=True, exist_ok=True)
    temporary = output_file.with_suffix(output_file.suffix + ".tmp")
    with h5py.File(temporary, "w") as target:
        relative_master = os.path.relpath(master_file, start=output_file.parent)
        target["entry1"] = h5py.ExternalLink(relative_master, "/entry1")
        target.attrs["creator"] = "I22 MoDaCor preprocessing notebook"
        target.attrs["source_file"] = relative_master
        target.attrs["preprocessing_version"] = PREPROCESSING_VERSION

        normalization = target.require_group("/modacor/normalization")
        normalization.attrs["description"] = "Frame-wise arrays reshaped for broadcasting over detector y/x axes."
        normalization.attrs["frame_shape"] = leading_shape
        normalization.attrs["bsdiodes_source"] = "/entry1/bsdiodes/data"
        normalization.attrs["bsdiodes_reduction_axis"] = 2
        normalization.attrs["bsdiodes_channel_index"] = BSDIODES_CHANNEL

        calibration = target.require_group("/modacor/calibration")
        calibration.attrs["description"] = "Scalar calibration values used by the I22 MoDaCor pipelines."
        _write_dataset(
            calibration,
            "absolute_intensity_factor",
            np.asarray(ABSOLUTE_INTENSITY_FACTOR, dtype=float),
            units="dimensionless",
            source="DAWN processing factor for this example",
        )

        _write_dataset(normalization, "bsdiodes_channel_1_mean", _as_detector_divisor(mean), units="dimensionless")
        _write_dataset(normalization, "bsdiodes_channel_1_std", _as_detector_divisor(std), units="dimensionless")
        _write_dataset(normalization, "bsdiodes_channel_1_sem", _as_detector_divisor(sem), units="dimensionless")
        _write_dataset(normalization, "bsdiodes_channel_1_n_valid", _as_detector_divisor(valid_count))
        _write_dataset(normalization, "transmission", _as_detector_divisor(transmission), units="dimensionless")

        for name, (values, units) in count_times.items():
            _write_dataset(normalization, name, _as_detector_divisor(values), units=units)

    temporary.replace(output_file)
    return output_file


# Keep this variable name for the later source-registration cells. These are the original
# DAWN/NeXus calibration files; MoDaCor now resolves their transformation chains directly.
preprocessed_calibration_files = dict(CALIBRATION_FILES)

measurement_inputs = list(sample_files)
if BACKGROUND_FILE.exists() and BACKGROUND_FILE not in measurement_inputs:
    measurement_inputs.append(BACKGROUND_FILE)
preprocessed_measurement_files = {
    source.resolve(): preprocess_i22_measurement(source, overwrite=OVERWRITE_PREPROCESSED)
    for source in measurement_inputs
}
preprocessed_sample_files = [preprocessed_measurement_files[path.resolve()] for path in sample_files]
preprocessed_background_file = preprocessed_measurement_files.get(BACKGROUND_FILE.resolve())

print("Calibration sources:")
for detector, path in preprocessed_calibration_files.items():
    with h5py.File(path, "r") as h5, h5py.File(MASK_FILES[detector], "r") as mask_h5:
        shape = tuple(h5["/entry1/calibration_data/data"].shape)
        mask_shape = tuple(mask_h5["/entry/mask/mask"].shape)
        if shape != mask_shape:
            raise ValueError(f"{detector} calibration shape {shape} does not match mask shape {mask_shape}.")
        detector_path = "/entry1/instrument/detector"
        depends_on = _decode(h5[f"{detector_path}/detector_module/module_offset"].attrs.get("depends_on", ""))
        print(f"  {detector}: {path}  shape={shape}, module_offset depends_on={depends_on!r}")

print(f"Preprocessed {len(preprocessed_sample_files)} sample measurement(s).")
if preprocessed_background_file is None:
    print(f"Background not preprocessed because it is missing: {BACKGROUND_FILE}")
else:
    print(f"Preprocessed background: {preprocessed_background_file}")


## Pipeline Graph Preview

This previews both detector pipelines. It does not require the sample or background files to be registered.


In [ ]:
from modacor.runner.pipeline import Pipeline

pipelines = {}
for detector, pipeline_path in PIPELINE_PATHS.items():
    pipeline = Pipeline.from_yaml_file(yaml_file=pipeline_path)
    pipeline.prepare()
    pipelines[detector] = pipeline
    mermaid_src = pipeline.to_mermaid(direction="TD")
    display(Markdown(f"### {detector}\n\n```mermaid\n{mermaid_src}\n```"))
    print(f"{detector}: {len(pipeline.graph)} configured step(s) from {pipeline_path}")

pipeline = pipelines[DETECTOR]
print(f"Selected pipeline: {DETECTOR} -> {PIPELINE_PATH}")

## Runtime API Helpers

These small helpers keep the HTTP calls readable while still showing which runtime endpoints are used.


In [ ]:
BASE_URLS = {
    detector: f"http://{SERVER_HOST}:{SERVER_PORTS[detector]}"
    for detector in DETECTORS_TO_RUN
}
if not SEPARATE_SERVER_PER_DETECTOR:
    BASE_URLS = {detector: f"http://{SERVER_HOST}:{SERVER_PORT}" for detector in DETECTORS_TO_RUN}
BASE_URL = BASE_URLS[DETECTOR]
SERVER_PROCESSES = {detector: None for detector in DETECTORS_TO_RUN}
SERVER_PROCESS = None


def api_url(path: str, detector: str | None = None) -> str:
    detector = detector or DETECTOR
    return BASE_URLS[detector].rstrip("/") + path


def api_request(
    method: str,
    path: str,
    *,
    detector: str | None = None,
    payload: dict | None = None,
    expected: tuple[int, ...] = (200, 201, 202, 204),
):
    response = requests.request(method, api_url(path, detector=detector), json=payload, timeout=120)
    if response.status_code not in expected:
        message = response.text
        try:
            message = json.dumps(response.json(), indent=2)
        except ValueError:
            pass
        raise RuntimeError(f"{method.upper()} {path} failed with HTTP {response.status_code}:" + "\n" + message)
    if response.status_code == 204 or not response.content:
        return None
    return response.json()


def readiness_ok(detector: str | None = None, timeout: float = 1.0) -> bool:
    try:
        response = requests.get(api_url("/v1/readiness", detector=detector), timeout=timeout)
        return response.ok and bool(response.json().get("ready", False))
    except requests.RequestException:
        return False


def display_text_block(text, *, title=None):
    if title:
        display(Markdown(f"**{title}**"))
    display(Markdown("\n".join(["```text", str(text).replace("```", "`` `"), "```"])))


## Start The Runtime Server

This starts a local MoDaCor server from the same Python environment as the notebook. If a server is already running on the configured port, the notebook reuses it.


In [ ]:
def _server_environment():
    server_env = os.environ.copy()
    if "hdf5plugin" in globals() and hasattr(hdf5plugin, "PLUGINS_PATH"):
        server_env.setdefault("HDF5_PLUGIN_PATH", hdf5plugin.PLUGINS_PATH)
    return server_env


def start_server(detector: str | None = None, timeout_s: float = 45.0):
    global SERVER_PROCESS
    detector = detector or DETECTOR
    base_url = BASE_URLS[detector]
    port = SERVER_PORTS[detector] if SEPARATE_SERVER_PER_DETECTOR else SERVER_PORT
    log_path = SERVER_LOG_PATHS[detector]

    if readiness_ok(detector, timeout=1.0):
        print(f"Runtime server for {detector} is already ready at {base_url}")
        print("If hdf5plugin was just installed, stop and restart this server before processing I22 detector data.")
        return None

    command = [
        sys.executable,
        "-m",
        "modacor.cli",
        "serve",
        "--host",
        SERVER_HOST,
        "--port",
        str(port),
    ]
    print(f"Starting {detector} runtime server:")
    print(" ".join(command))

    server_log_file = open(log_path, "a", buffering=1)

    process = subprocess.Popen(
        command,
        stdout=server_log_file,
        stderr=subprocess.STDOUT,
        env=_server_environment(),
        text=True,
    )
    SERVER_PROCESSES[detector] = process
    if detector == DETECTOR:
        SERVER_PROCESS = process

    deadline = time.monotonic() + timeout_s
    while time.monotonic() < deadline:
        if readiness_ok(detector, timeout=1.0):
            print(f"Runtime server for {detector} ready at {base_url}")
            return process
        if process.poll() is not None:
            raise RuntimeError(
                f"Runtime server for {detector} exited early with code {process.returncode}. "
                f"See {log_path}."
            )
        time.sleep(0.5)

    raise TimeoutError(f"Runtime server for {detector} did not become ready within {timeout_s:.0f} seconds at {base_url}")


def start_servers(timeout_s: float = 45.0):
    detectors = DETECTORS_TO_RUN if SEPARATE_SERVER_PER_DETECTOR else [DETECTOR]
    return {detector: start_server(detector, timeout_s=timeout_s) for detector in detectors}


def stop_server(detector: str | None = None):
    global SERVER_PROCESS
    detectors = [detector] if detector is not None else list(SERVER_PROCESSES.keys())

    for detector_name in detectors:
        process = SERVER_PROCESSES.get(detector_name)
        if process is None:
            print(f"No notebook-owned {detector_name} server process to stop.")
            continue
        if process.poll() is not None:
            print(f"{detector_name} server process already exited with code {process.returncode}.")
            SERVER_PROCESSES[detector_name] = None
            continue

        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait(timeout=10)
        print(f"Stopped notebook-owned {detector_name} runtime server.")
        SERVER_PROCESSES[detector_name] = None
    SERVER_PROCESS = None


atexit.register(stop_server)

start_servers()
readiness = {
    detector: api_request("GET", "/v1/readiness", detector=detector)
    for detector in (DETECTORS_TO_RUN if SEPARATE_SERVER_PER_DETECTOR else [DETECTOR])
}
display(JSON(readiness))


## Create A Fresh Runtime Session

This deletes an existing session with the same ID, then creates a clean one from the current pipeline YAML. Re-run this cell after changing the pipeline YAML or tracer settings.


In [ ]:
def delete_session_if_exists(detector: str, session_id: str):
    response = requests.delete(api_url(f"/v1/sessions/{session_id}", detector=detector), timeout=30)
    if response.status_code == 204:
        print(f"Deleted existing session: {session_id}")
        return
    if response.status_code == 404:
        print(f"No existing session named {session_id!r}.")
        return
    raise RuntimeError(f"DELETE session failed with HTTP {response.status_code}: {response.text}")


def create_server_session(detector: str):
    session_id = SESSION_IDS[detector]
    delete_session_if_exists(detector, session_id)
    session_payload = {
        "session_id": session_id,
        "name": f"I22 {detector} server batch",
        "pipeline": {"yaml_path": str(PIPELINE_PATHS[detector])},
        "trace": {
            "enabled": TRACE_ENABLED,
            "watch": TRACE_WATCH if TRACE_ENABLED else {},
            "record_only_on_change": True,
            "snapshot_processing_data": False,
            "snapshot_step_ids": [],
        },
        "auto_full_reset_on_partial_error": True,
    }
    return api_request("POST", "/v1/sessions", detector=detector, payload=session_payload, expected=(200, 201))


sessions = {detector: create_server_session(detector) for detector in DETECTORS_TO_RUN}
display(JSON(sessions))


## Register Sources

The session receives six explicit source roles: `sample`, `background`, `saxs_calibration`, `saxs_mask`,
`waxs_calibration`, and `waxs_mask`. Only `sample` changes inside the batch loop.


In [ ]:
def build_source_registrations(detector: str, sample_file_for_sample=None):
    background_file = BACKGROUND_FILES[detector]
    sources_to_register = [
        {
            "ref": "saxs_calibration",
            "type": "hdf",
            "location": str(preprocessed_calibration_files["SAXS"]),
        },
        {"ref": "saxs_mask", "type": "hdf", "location": str(MASK_FILES["SAXS"])},
        {
            "ref": "waxs_calibration",
            "type": "hdf",
            "location": str(preprocessed_calibration_files["WAXS"]),
        },
        {"ref": "waxs_mask", "type": "hdf", "location": str(MASK_FILES["WAXS"])},
    ]

    background_source = preprocessed_measurement_files.get(background_file.resolve())
    if background_source is not None:
        sources_to_register.append(
            {"ref": "background", "type": "hdf", "location": str(background_source)}
        )
    else:
        print(f"Background file missing, not registering it yet: {background_file}")

    if sample_file_for_sample is None and preprocessed_sample_files:
        sample_file_for_sample = preprocessed_sample_files[0]
    if sample_file_for_sample is not None:
        sources_to_register.append(
            {"ref": "sample", "type": "hdf", "location": str(sample_file_for_sample)}
        )
    else:
        print("No preprocessed sample file is available yet.")

    return sources_to_register


def register_session_sources(detector: str, sample_file_for_sample=None, *, display_result=True):
    session_id = SESSION_IDS[detector]
    sources_to_register = build_source_registrations(detector, sample_file_for_sample=sample_file_for_sample)
    registered_sources = api_request(
        "PUT",
        f"/v1/sessions/{session_id}/sources",
        detector=detector,
        payload={"sources": sources_to_register},
    )
    if display_result:
        display(JSON(registered_sources))
    return registered_sources


def register_session_plot_sink(detector: str, *, display_result=True):
    session_id = SESSION_IDS[detector]
    registered_sinks = api_request(
        "PUT",
        f"/v1/sessions/{session_id}/sinks",
        detector=detector,
        payload={
            "sinks": [
                {"ref": PLOT_SINK_REF, "type": "plotly_json", "location": "buffer://session"}
            ]
        },
    )
    if display_result:
        display(JSON(registered_sinks))
    return registered_sinks


def register_session_result_hdf_sink(detector: str, output_path: Path, *, display_result=True):
    session_id = SESSION_IDS[detector]
    registered_sink = api_request(
        "POST",
        f"/v1/sessions/{session_id}/sinks/patch",
        detector=detector,
        payload={
            "ref": RESULT_HDF_SINK_REF,
            "type": "hdf",
            "location": str(output_path),
        },
    )
    if display_result:
        display(JSON(registered_sink))
    return registered_sink


def reset_session_after_failed_sample(detector: str):
    print(f"Resetting the {detector} server session after a failed sample because rollback snapshots are disabled.")
    session = create_server_session(detector)
    registered_sources = register_session_sources(detector, display_result=False)
    registered_sinks = register_session_plot_sink(detector, display_result=False)
    print(f"{detector} session reset complete. The next sample will run in full mode to seed fresh state.")
    return {"session": session, "sources": registered_sources, "sinks": registered_sinks}


registered_sources = {
    detector: register_session_sources(detector, display_result=False)
    for detector in DETECTORS_TO_RUN
}
registered_plot_sinks = {
    detector: register_session_plot_sink(detector, display_result=False)
    for detector in DETECTORS_TO_RUN
}
display(JSON({"sources": registered_sources, "plot_sinks": registered_plot_sinks}))


## Session Status

This optional check shows the current session, registered sources, and trace configuration.


In [ ]:
session_status = {
    detector: api_request("GET", f"/v1/sessions/{SESSION_IDS[detector]}", detector=detector)
    for detector in DETECTORS_TO_RUN
}
display(JSON(session_status))


## Live Server Plots

These links open auto-refreshing Plotly pages served by the MoDaCor runtime. They show the latest plot payload published by the pipeline after each processed sample.


In [ ]:
def live_plot_url(detector: str, plot_id: str) -> str:
    session_id = SESSION_IDS[detector]
    return api_url(f"/v1/sessions/{session_id}/plots/{PLOT_SINK_REF}/{plot_id}", detector=detector)


def display_live_plot_links():
    lines = []
    for detector in DETECTORS_TO_RUN:
        plot_ids = LIVE_PLOT_IDS[detector]
        lines.append(f"### {detector}")
        lines.append(f"- [1D corrected I(Q)]({live_plot_url(detector, plot_ids['1d'])})")
        lines.append(f"- [2D corrected detector image]({live_plot_url(detector, plot_ids['2d'])})")
    display(Markdown("\n".join(lines)))


display_live_plot_links()


## Batch Processing Loop

The first processed sample runs in `full` mode to seed runtime state. Later samples update only the `sample` source and run in `auto` mode, so the server reuses unchanged state where possible.

With `ROLLBACK_SNAPSHOT = False`, failed partial runs are handled by recreating the session before continuing. The next sample then runs in `full` mode once to seed clean state again.


In [ ]:
# limit number of sample files to the first 10 for testing purposes only:
sample_files = sample_files[:10]
preprocessed_sample_files = preprocessed_sample_files[:10]

In [ ]:
run_results = []
failed_results = []
session_has_processing_state = {detector: False for detector in DETECTORS_TO_RUN}


def process_detector_sample(detector: str, index: int, sample_file: Path, preprocessed_sample_file: Path):
    session_id = SESSION_IDS[detector]
    run_name = f"{sample_file.stem}_{detector.lower()}"
    output_path = OUTPUT_DIR / f"{sample_file.stem}_{detector.lower()}_server_result.h5"
    mode = "auto" if session_has_processing_state[detector] else "full"

    try:
        api_request(
            "PUT",
            f"/v1/sessions/{session_id}/sources",
            detector=detector,
            payload={
                "sources": [
                    {"ref": "sample", "type": "hdf", "location": str(preprocessed_sample_file)}
                ]
            },
        )

        register_session_result_hdf_sink(detector, output_path, display_result=False)

        process_payload = {
            "mode": mode,
            "run_name": run_name,
            "rollback_snapshot": ROLLBACK_SNAPSHOT,
            "write_hdf": {
                "path": str(output_path),
                "data_paths": SAMPLE_OUTPUT_DATA_PATHS,
            },
        }
        if mode == "auto":
            process_payload["changed_sources"] = ["sample"]

        result = api_request(
            "POST",
            f"/v1/sessions/{session_id}/process",
            detector=detector,
            payload=process_payload,
        )
    except Exception as exc:
        latest_error = None
        diagnostics_error = None
        try:
            latest_error = api_request("GET", f"/v1/sessions/{session_id}/errors/latest", detector=detector)
        except Exception as diagnostics_exc:
            diagnostics_error = str(diagnostics_exc)

        return {
            "ok": False,
            "detector": detector,
            "index": index,
            "sample": str(sample_file),
            "preprocessed_sample": str(preprocessed_sample_file),
            "output": str(output_path),
            "mode": mode,
            "error": str(exc),
            "latest_error": latest_error,
            "diagnostics_error": diagnostics_error,
        }

    return {
        "ok": True,
        "detector": detector,
        "index": index,
        "sample": str(sample_file),
        "preprocessed_sample": str(preprocessed_sample_file),
        "output": str(output_path),
        "mode": mode,
        "result": result,
    }


def process_sample_for_detectors(index: int, sample_file: Path, preprocessed_sample_file: Path):
    if PARALLEL_DETECTOR_RUNS and len(DETECTORS_TO_RUN) > 1:
        with ThreadPoolExecutor(max_workers=len(DETECTORS_TO_RUN)) as executor:
            futures = {
                executor.submit(process_detector_sample, detector, index, sample_file, preprocessed_sample_file): detector
                for detector in DETECTORS_TO_RUN
            }
            results = [future.result() for future in as_completed(futures)]
        return sorted(results, key=lambda item: DETECTORS_TO_RUN.index(item["detector"]))

    return [
        process_detector_sample(detector, index, sample_file, preprocessed_sample_file)
        for detector in DETECTORS_TO_RUN
    ]


if not sample_files:
    print("No sample files found yet. Rerun discovery and preprocessing after copying finishes.")
else:
    missing_backgrounds = [detector for detector in DETECTORS_TO_RUN if BACKGROUND_FILES[detector].resolve() not in preprocessed_measurement_files]
    if missing_backgrounds:
        missing = {detector: str(BACKGROUND_FILES[detector]) for detector in missing_backgrounds}
        raise FileNotFoundError(f"Background file(s) missing: {missing}")

    sample_batch = list(zip(sample_files, preprocessed_sample_files, strict=True))
    if MAX_SAMPLES_TO_PROCESS is not None:
        sample_batch = sample_batch[:MAX_SAMPLES_TO_PROCESS]
    print(f"Detector concurrency: {'on' if PARALLEL_DETECTOR_RUNS else 'off'}")
    print(f"Server mode: {'separate detector servers' if SEPARATE_SERVER_PER_DETECTOR else 'single shared server'}")
    print(f"Processing {len(sample_batch)} of {len(sample_files)} discovered sample file(s).")
    for index, (sample_file, preprocessed_sample_file) in enumerate(sample_batch):
        print(f"\n[{index + 1}/{len(sample_batch)}] {sample_file.name}")
        print(f"Source: {preprocessed_sample_file}")

        sample_results = process_sample_for_detectors(index, sample_file, preprocessed_sample_file)
        sample_failures = []
        for item in sample_results:
            detector = item["detector"]
            if item["ok"]:
                session_has_processing_state[detector] = True
                run_results.append({key: value for key, value in item.items() if key != "ok"})
                result = item["result"]
                print(
                    f"{detector}: {result.get('status')} mode={item['mode']} "
                    f"effective_mode={result.get('effective_mode')} run_id={result.get('run_id')}"
                )
                continue

            sample_failures.append(item)
            failed_results.append({key: value for key, value in item.items() if key != "ok"})
            print(f"{detector}: failed mode={item['mode']} output={item['output']}")
            print(item["error"])
            if item.get("diagnostics_error"):
                print("Could not fetch latest server diagnostics:", item["diagnostics_error"])
            if item.get("latest_error"):
                display(JSON(item["latest_error"]))

        if sample_failures:
            failed_server_down = any(not readiness_ok(item["detector"], timeout=2.0) for item in sample_failures)
            if not CONTINUE_ON_SAMPLE_ERROR or failed_server_down:
                failed = ", ".join(item["detector"] for item in sample_failures)
                raise RuntimeError(f"Detector run(s) failed for {sample_file.name}: {failed}")

            if RESET_SESSION_AFTER_FAILURE:
                for item in sample_failures:
                    reset_session_after_failed_sample(item["detector"])
                    session_has_processing_state[item["detector"]] = False
            print("Continuing with the next sample.")

summary = {"succeeded": run_results, "failed": failed_results}
print(f"Succeeded: {len(run_results)}  Failed: {len(failed_results)}")
display(JSON(summary))


## Optional Diagnostics

Run this only when something looks wrong. Trace reports are available only when `TRACE_ENABLED = True` before creating the session.


In [ ]:
for detector in DETECTORS_TO_RUN:
    session_id = SESSION_IDS[detector]
    display(Markdown(f"### {detector}"))
    if TRACE_ENABLED:
        runs_payload = api_request("GET", f"/v1/sessions/{session_id}/runs", detector=detector)
        runs = runs_payload.get("runs", [])
        latest_run = runs[-1] if runs else None
        trace_report = latest_run.get("trace_report") if latest_run else None

        if not trace_report:
            latest_error = api_request("GET", f"/v1/sessions/{session_id}/errors/latest", detector=detector)
            error_payload = latest_error.get("current_error") or latest_error.get("latest_error") or {}
            trace_report = (error_payload.get("details") or {}).get("trace_report")

        if trace_report:
            display_text_block(trace_report, title="Tracer report")
        else:
            print("No trace report recorded for this session yet.")
    else:
        print("Tracing is disabled. Set TRACE_ENABLED = True and recreate the session to collect trace reports.")

    latest_error = api_request("GET", f"/v1/sessions/{session_id}/errors/latest", detector=detector)
    error_payload = latest_error.get("current_error") or latest_error.get("latest_error") or {}
    error_details = error_payload.get("details") or {}

    if error_payload:
        display(JSON(latest_error))
        if error_details.get("traceback"):
            display_text_block(error_details["traceback"], title="Traceback")
    else:
        print("No server error is currently recorded for this session.")


## Stop The Server

Run this when you are done. The notebook also registers an automatic cleanup hook for a server process it started itself.


In [ ]:
stop_server()


## Plot A Processed Output File

This final cell loads one server output HDF5 file and plots the final sample signal against Q. It uses x error bars from Q uncertainty when available, and y error bars from signal uncertainty when available.


In [ ]:
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np


def _decode_hdf_attr(value):
    if isinstance(value, bytes):
        return value.decode("utf-8")
    if isinstance(value, np.ndarray) and value.shape == ():
        return _decode_hdf_attr(value.item())
    return value


def _latest_output_file():
    if "run_results" in globals():
        successful_outputs = [Path(item["output"]) for item in run_results if Path(item["output"]).exists()]
        if successful_outputs:
            return successful_outputs[-1]

    output_files = sorted(OUTPUT_DIR.glob("*_server_result.h5"), key=lambda item: item.stat().st_mtime)
    if not output_files:
        raise FileNotFoundError(f"No server output files found in {OUTPUT_DIR}")
    return output_files[-1]


def _default_child(group):
    default_name = _decode_hdf_attr(group.attrs.get("default"))
    if default_name in group:
        return group[default_name], str(default_name)
    first_name = next(iter(group.keys()))
    return group[first_name], str(first_name)


def _uncertainty(group, preferred_names):
    uncertainties = group.get("uncertainties")
    if uncertainties is None:
        return None, None
    for name in preferred_names:
        if name in uncertainties:
            return uncertainties[name][()], name
    return None, None


plot_output_path = _latest_output_file()

with h5py.File(plot_output_path, "r") as h5:
    result_group, run_name = _default_child(h5["processing/result"])
    sample_group = result_group["sample"]
    signal_group = sample_group["signal"]

    q = signal_group["Q"][()] if "Q" in signal_group else sample_group["Q"]["signal"][()]
    signal = signal_group["signal"][()]
    q_units = _decode_hdf_attr(signal_group.get("Q", sample_group["Q"]["signal"]).attrs.get("units", ""))
    signal_units = _decode_hdf_attr(signal_group["signal"].attrs.get("units", ""))

    xerr = None
    xerr_name = None
    if "Q" in sample_group:
        xerr, xerr_name = _uncertainty(sample_group["Q"], ["uncertainty_combined", "SEM", "STD"])

    yerr, yerr_name = _uncertainty(signal_group, ["uncertainty_pixelvalues", "SEM", "STD", "poisson"])

valid = np.isfinite(q) & np.isfinite(signal)
if xerr is not None:
    valid &= np.isfinite(xerr)
if yerr is not None:
    valid &= np.isfinite(yerr)

fig, ax = plt.subplots(figsize=(7.5, 4.8), constrained_layout=True)
ax.errorbar(
    q[valid],
    signal[valid],
    xerr=xerr[valid] if xerr is not None else None,
    yerr=yerr[valid] if yerr is not None else None,
    fmt="o",
    linestyle="none",
    markersize=3,
    elinewidth=0.7,
    capsize=1.5,
    alpha=0.8,
)
if np.all(q[valid] > 0):
    ax.set_xscale("log")
if np.all(signal[valid] > 0):
    ax.set_yscale("log")
ax.set_xlabel(f"Q ({q_units})" if q_units else "Q")
ax.set_ylabel(f"Signal ({signal_units})" if signal_units else "Signal")
ax.set_title(plot_output_path.name)
ax.grid(True, which="both", alpha=0.25)

print(f"Loaded: {plot_output_path}")
print(f"Run: {run_name}")
print(f"x error: {xerr_name or 'not available'}")
print(f"y error: {yerr_name or 'not available'}")
plt.show()
